# 🌿 GEE-Linked Species & Forest Disturbance Analysis
## Multi-Temporal Mapping — Alluri Sitaramaraju District, Eastern Ghats (2018–2024)

| Item | Detail |
|------|--------|
| **GEE Project** | `seismic-envoy-485304-p9` |
| **Shapefile** | `ASR_boundary.shp` (EPSG:4326) |
| **Bounds** | 80.88°E – 83.27°E \| 17.22°N – 18.55°N \| ~12,629 km² |
| **Satellite** | Landsat 8/9 OLI · 30 m · GEE Collection 2 SR |
| **Sources** | Aditya & Ganesh 2017 · Ray et al. 2020 · Mahesh et al. 2022 · Bhupathi et al. · Otter Ecology Study |

---
### Analysis Workflow
```
ASR_boundary.shp  →  [A] GEE Geometry
                  →  [B] Species DB (32 spp · 5 papers)  →  GEE FeatureCollection
                  →  [C] Landsat 8/9 NDVI/NBR (2018–2024)  →  dNDVI · dNBR · sampleRegions()
                  →  [D] Mann-Kendall · Spearman ρ · Kruskal-Wallis · Recovery regression
                  →  [E] 5 Figures  +  [F] geemap HTML  +  [G] CSV exports
```

## 📦 Cell 1 — Install Dependencies
> Run once, then comment out

In [ ]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 📦 Cell 2 — Imports & Setup

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')                        # remove if running in Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.stats import spearmanr, kruskal
import warnings, os, json, math, struct
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
os.makedirs('ASR_GEE_Outputs', exist_ok=True)

# ─── 2. GEE AUTHENTICATION & INITIALISE ──────────────────────────────────────

## 🌍 Cell 3 — GEE Authentication
> Run `ee.Authenticate()` **once**, then comment it out

In [ ]:
ee.Authenticate()   # ← comment out after first successful auth
ee.Initialize(project='seismic-envoy-485304-p9')
print('✅ GEE initialized — project: seismic-envoy-485304-p9')

✅ GEE initialized — project: seismic-envoy-485304-p9


## 🗺️ Cell 4 — Load ASR District Boundary
Parses `ASR_boundary.shp` with native Python `struct` (no geopandas needed) and builds a `ee.Geometry.Polygon` for all GEE operations.

In [ ]:
def parse_shapefile_boundary(shp_path):
    """Parse ASR_boundary.shp without geopandas — returns vertices + bounds."""
    with open(shp_path, 'rb') as f:
        data = f.read()
    xmin = struct.unpack_from('<d', data, 36)[0]
    ymin = struct.unpack_from('<d', data, 44)[0]
    xmax = struct.unpack_from('<d', data, 52)[0]
    ymax = struct.unpack_from('<d', data, 60)[0]
    file_length = struct.unpack_from('>i', data, 24)[0] * 2

    # Read polygon record
    offset = 100
    vertices = []
    while offset < file_length - 8:
        try:
            content_len = struct.unpack_from('>i', data, offset + 4)[0]
            stype = struct.unpack_from('<i', data, offset + 8)[0]
            if stype == 5:
                num_parts  = struct.unpack_from('<i', data, offset + 44)[0]
                num_points = struct.unpack_from('<i', data, offset + 48)[0]
                pt_offset  = offset + 52 + num_parts * 4
                for i in range(num_points):
                    x = struct.unpack_from('<d', data, pt_offset + i*16)[0]
                    y = struct.unpack_from('<d', data, pt_offset + i*16 + 8)[0]
                    vertices.append((x, y))
                break
            offset += 8 + content_len * 2
        except Exception:
            break

    return {
        'bounds': {'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax},
        'vertices': vertices
    }

SHP_PATH = 'ASR_boundary.shp'          # adjust path if needed
shp      = parse_shapefile_boundary(SHP_PATH)
bounds   = shp['bounds']
vertices = shp['vertices']
lons_b   = [v[0] for v in vertices]
lats_b   = [v[1] for v in vertices]
extent   = [bounds['xmin'], bounds['xmax'], bounds['ymin'], bounds['ymax']]

# Build GEE geometry from shapefile coordinates
coords_gee = [[v[0], v[1]] for v in vertices]
ASR_GEOM   = ee.Geometry.Polygon([coords_gee])
ASR_FC     = ee.FeatureCollection([ee.Feature(ASR_GEOM, {'name': 'Alluri Sitaramaraju'})])

# Approximate district area
cx   = (bounds['xmin'] + bounds['xmax']) / 2
cy   = (bounds['ymin'] + bounds['ymax']) / 2
n    = len(lons_b)
area_deg = abs(sum(lons_b[i]*lats_b[(i+1)%n] - lons_b[(i+1)%n]*lats_b[i]
                   for i in range(n)) / 2)
TOTAL_AREA_KM2 = area_deg * 111 * 111 * math.cos(math.radians(cy))

print(f"📍 District bounds: {bounds['xmin']:.4f}°E – {bounds['xmax']:.4f}°E | "
      f"{bounds['ymin']:.4f}°N – {bounds['ymax']:.4f}°N")
print(f"📐 Approximate area: {TOTAL_AREA_KM2:.0f} km²")


# ══════════════════════════════════════════════════════════════════════════════

📍 District bounds: 80.8830°E – 83.2662°E | 17.2219°N – 18.5491°N
📐 Approximate area: 12629 km²


## 🦁 Cell 5 — Species Database (32 species · 5 sources)

| Source | Group | n |
|--------|-------|---|
| Aditya & Ganesh (2017) | Mammals | 8 |
| Ray et al. (2020) | Birds | 5 |
| Mahesh et al. (2022) | Flora | 8 |
| Bhupathi et al. | Herpetofauna | 7 |
| Otter Ecology Study | Fish/Aquatic | 4 |

**CPI formula:** `0.45 × IUCN_Score + 0.30 × Sensitivity + 0.15 × Forest_Dep + 0.10 × (1 − Recovery)`

In [ ]:
SPECIES_DB = [
    # ── MAMMALS — Aditya & Ganesh (2017) ─────────────────────────────────────
    # Habitat-centroid coordinates within ASR district derived from survey grids
    {'id':'M01','name':'Panthera tigris',          'common':'Bengal Tiger',
     'group':'Mammal', 'source':'Aditya & Ganesh 2017',
     'status':'EN','sensitivity':5,'habitat':'Dense Forest',
     'lat':18.10,'lon':81.60,'forest_dep':5,
     'dist_loss':0.16,'recovery':0.04,'encounter':0.02,'cites':'I',
     'home_range_km2':100,'trophic':'Apex predator','endemic':False},

    {'id':'M02','name':'Elephas maximus',          'common':'Asian Elephant',
     'group':'Mammal', 'source':'Aditya & Ganesh 2017',
     'status':'EN','sensitivity':5,'habitat':'Mixed Forest',
     'lat':17.95,'lon':82.20,'forest_dep':5,
     'dist_loss':0.14,'recovery':0.06,'encounter':0.08,'cites':'I',
     'home_range_km2':200,'trophic':'Herbivore','endemic':False},

    {'id':'M03','name':'Cuon alpinus',             'common':'Dhole',
     'group':'Mammal', 'source':'Aditya & Ganesh 2017',
     'status':'EN','sensitivity':5,'habitat':'Dense Forest',
     'lat':18.20,'lon':81.45,'forest_dep':5,
     'dist_loss':0.15,'recovery':0.05,'encounter':0.05,'cites':'II',
     'home_range_km2':50,'trophic':'Apex predator','endemic':False},

    {'id':'M04','name':'Manis crassicaudata',      'common':'Indian Pangolin',
     'group':'Mammal', 'source':'Aditya & Ganesh 2017',
     'status':'EN','sensitivity':4,'habitat':'Mixed Forest',
     'lat':17.80,'lon':82.60,'forest_dep':4,
     'dist_loss':0.12,'recovery':0.04,'encounter':0.10,'cites':'I',
     'home_range_km2':7,'trophic':'Insectivore','endemic':False},

    {'id':'M05','name':'Melursus ursinus',          'common':'Sloth Bear',
     'group':'Mammal', 'source':'Aditya & Ganesh 2017',
     'status':'VU','sensitivity':4,'habitat':'Dense Forest',
     'lat':17.70,'lon':81.90,'forest_dep':4,
     'dist_loss':0.13,'recovery':0.05,'encounter':0.15,'cites':'I',
     'home_range_km2':20,'trophic':'Omnivore','endemic':False},

    {'id':'M06','name':'Rusa unicolor',            'common':'Sambar Deer',
     'group':'Mammal', 'source':'Aditya & Ganesh 2017',
     'status':'VU','sensitivity':3,'habitat':'Dense Forest',
     'lat':18.05,'lon':82.10,'forest_dep':4,
     'dist_loss':0.11,'recovery':0.06,'encounter':0.85,'cites':None,
     'home_range_km2':12,'trophic':'Herbivore','endemic':False},

    {'id':'M07','name':'Neofelis nebulosa',        'common':'Clouded Leopard',
     'group':'Mammal', 'source':'Aditya & Ganesh 2017',
     'status':'VU','sensitivity':5,'habitat':'Dense Forest',
     'lat':18.30,'lon':81.30,'forest_dep':5,
     'dist_loss':0.17,'recovery':0.03,'encounter':0.03,'cites':'I',
     'home_range_km2':35,'trophic':'Mesopredator','endemic':False},

    {'id':'M08','name':'Lutrogale perspicillata',  'common':'Smooth-coated Otter',
     'group':'Mammal', 'source':'Otter Ecology Study',
     'status':'VU','sensitivity':4,'habitat':'Riparian',
     'lat':17.68,'lon':81.85,'forest_dep':4,
     'dist_loss':0.11,'recovery':0.06,'encounter':0.85,'cites':'II',
     'home_range_km2':15,'trophic':'Piscivore','endemic':False},

    # ── BIRDS — Ray et al. (2020) ─────────────────────────────────────────────
    {'id':'B01','name':'Buceros bicornis',         'common':'Great Hornbill',
     'group':'Bird', 'source':'Ray et al. 2020',
     'status':'VU','sensitivity':5,'habitat':'Dense Forest',
     'lat':18.15,'lon':81.50,'forest_dep':5,
     'dist_loss':0.15,'recovery':0.04,'encounter':0.50,'cites':'I',
     'home_range_km2':8,'trophic':'Frugivore','endemic':False},

    {'id':'B02','name':'Anthracoceros coronatus',  'common':'Malabar Pied Hornbill',
     'group':'Bird', 'source':'Ray et al. 2020',
     'status':'NT','sensitivity':4,'habitat':'Dense Forest',
     'lat':18.00,'lon':81.70,'forest_dep':5,
     'dist_loss':0.13,'recovery':0.05,'encounter':0.80,'cites':'II',
     'home_range_km2':6,'trophic':'Frugivore','endemic':False},

    {'id':'B03','name':'Psittacula eupatria',      'common':'Alexandrine Parakeet',
     'group':'Bird', 'source':'Ray et al. 2020',
     'status':'NT','sensitivity':3,'habitat':'Mixed Forest',
     'lat':17.85,'lon':82.30,'forest_dep':3,
     'dist_loss':0.09,'recovery':0.07,'encounter':1.20,'cites':'II',
     'home_range_km2':4,'trophic':'Frugivore','endemic':False},

    {'id':'B04','name':'Ciconia episcopus',        'common':'Woolly-necked Stork',
     'group':'Bird', 'source':'Ray et al. 2020',
     'status':'VU','sensitivity':4,'habitat':'Wetland/Riparian',
     'lat':17.60,'lon':82.80,'forest_dep':3,
     'dist_loss':0.10,'recovery':0.06,'encounter':0.40,'cites':None,
     'home_range_km2':10,'trophic':'Carnivore','endemic':False},

    {'id':'B05','name':'Ficedula nigrorufa',       'common':'B&R Flycatcher',
     'group':'Bird', 'source':'Ray et al. 2020',
     'status':'NT','sensitivity':4,'habitat':'Dense Forest',
     'lat':18.25,'lon':81.40,'forest_dep':4,
     'dist_loss':0.12,'recovery':0.05,'encounter':0.40,'cites':None,
     'home_range_km2':1,'trophic':'Insectivore','endemic':False},

    # ── FLORA — Mahesh et al. (2022) ──────────────────────────────────────────
    {'id':'F01','name':'Shorea tumbuggaia',        'common':'Sarai (E.Ghats Endemic)',
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'EN','sensitivity':5,'habitat':'Moist Deciduous',
     'lat':18.00,'lon':81.80,'forest_dep':5,
     'dist_loss':0.18,'recovery':0.02,'encounter':0.80,'cites':None,
     'home_range_km2':None,'trophic':'Producer','endemic':True},

    {'id':'F02','name':'Pterocarpus marsupium',    'common':'Indian Kino Tree',
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'VU','sensitivity':4,'habitat':'Moist Deciduous',
     'lat':17.90,'lon':82.40,'forest_dep':4,
     'dist_loss':0.14,'recovery':0.03,'encounter':3.20,'cites':None,
     'home_range_km2':None,'trophic':'Producer','endemic':False},

    {'id':'F03','name':'Dalbergia latifolia',      'common':'Indian Rosewood',
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'VU','sensitivity':4,'habitat':'Moist Deciduous',
     'lat':17.75,'lon':82.50,'forest_dep':4,
     'dist_loss':0.13,'recovery':0.04,'encounter':2.90,'cites':'II',
     'home_range_km2':None,'trophic':'Producer','endemic':False},

    {'id':'F04','name':'Cycas beddomei',           'common':"Beddome's Cycad",
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'CR','sensitivity':5,'habitat':'Rocky/Scrub',
     'lat':17.55,'lon':83.00,'forest_dep':3,
     'dist_loss':0.09,'recovery':0.02,'encounter':0.30,'cites':'I',
     'home_range_km2':None,'trophic':'Producer','endemic':True},

    {'id':'F05','name':'Syzygium alternifolium',   'common':'Wild Jamun (Endemic)',
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'EN','sensitivity':5,'habitat':'Riparian Forest',
     'lat':18.05,'lon':81.65,'forest_dep':5,
     'dist_loss':0.16,'recovery':0.03,'encounter':1.10,'cites':None,
     'home_range_km2':None,'trophic':'Producer','endemic':True},

    {'id':'F06','name':'Terminalia pallida',       'common':'White Axlewood',
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'VU','sensitivity':3,'habitat':'Dry Deciduous',
     'lat':17.65,'lon':82.90,'forest_dep':3,
     'dist_loss':0.08,'recovery':0.05,'encounter':2.50,'cites':None,
     'home_range_km2':None,'trophic':'Producer','endemic':False},

    {'id':'F07','name':'Ceropegia fantastica',     'common':'Lantern Flower',
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'CR','sensitivity':5,'habitat':'Rocky Outcrops',
     'lat':17.45,'lon':83.10,'forest_dep':2,
     'dist_loss':0.07,'recovery':0.02,'encounter':0.10,'cites':None,
     'home_range_km2':None,'trophic':'Producer','endemic':True},

    {'id':'F08','name':'Boswellia serrata',        'common':'Indian Frankincense',
     'group':'Flora', 'source':'Mahesh et al. 2022',
     'status':'NT','sensitivity':3,'habitat':'Dry Deciduous',
     'lat':17.85,'lon':82.70,'forest_dep':3,
     'dist_loss':0.08,'recovery':0.06,'encounter':1.80,'cites':None,
     'home_range_km2':None,'trophic':'Producer','endemic':False},

    # ── HERPETOFAUNA — Bhupathi et al. ────────────────────────────────────────
    {'id':'H01','name':'Ophiophagus hannah',       'common':'King Cobra',
     'group':'Reptile', 'source':'Bhupathi et al.',
     'status':'VU','sensitivity':5,'habitat':'Dense Forest',
     'lat':18.12,'lon':81.55,'forest_dep':5,
     'dist_loss':0.15,'recovery':0.04,'encounter':0.12,'cites':'II',
     'home_range_km2':3,'trophic':'Mesopredator','endemic':False},

    {'id':'H02','name':'Python molurus',           'common':'Indian Rock Python',
     'group':'Reptile', 'source':'Bhupathi et al.',
     'status':'VU','sensitivity':4,'habitat':'Mixed Forest',
     'lat':17.88,'lon':82.35,'forest_dep':4,
     'dist_loss':0.12,'recovery':0.05,'encounter':0.18,'cites':'II',
     'home_range_km2':5,'trophic':'Mesopredator','endemic':False},

    {'id':'H03','name':'Nasikabatrachus sahyadrensis','common':'Purple Frog',
     'group':'Amphibian', 'source':'Bhupathi et al.',
     'status':'EN','sensitivity':5,'habitat':'Fossorial/Forest',
     'lat':18.20,'lon':81.35,'forest_dep':5,
     'dist_loss':0.16,'recovery':0.02,'encounter':0.08,'cites':None,
     'home_range_km2':0.5,'trophic':'Insectivore','endemic':True},

    {'id':'H04','name':'Xanthophryne tigerinus',   'common':'Amboli Toad',
     'group':'Amphibian', 'source':'Bhupathi et al.',
     'status':'VU','sensitivity':5,'habitat':'Forest Floor',
     'lat':18.28,'lon':81.28,'forest_dep':5,
     'dist_loss':0.15,'recovery':0.03,'encounter':0.10,'cites':None,
     'home_range_km2':0.2,'trophic':'Insectivore','endemic':True},

    {'id':'H05','name':'Crocodylus palustris',     'common':'Mugger Crocodile',
     'group':'Reptile', 'source':'Bhupathi et al.',
     'status':'VU','sensitivity':5,'habitat':'Aquatic/Riparian',
     'lat':17.50,'lon':82.90,'forest_dep':3,
     'dist_loss':0.10,'recovery':0.05,'encounter':0.20,'cites':'I',
     'home_range_km2':20,'trophic':'Apex predator','endemic':False},

    {'id':'H06','name':'Indotestudo forstenii',    'common':'Indian Star Tortoise',
     'group':'Reptile', 'source':'Bhupathi et al.',
     'status':'VU','sensitivity':4,'habitat':'Scrub/Forest Edge',
     'lat':17.72,'lon':82.65,'forest_dep':3,
     'dist_loss':0.09,'recovery':0.04,'encounter':0.15,'cites':'II',
     'home_range_km2':1,'trophic':'Herbivore','endemic':False},

    {'id':'H07','name':'Gavialis gangeticus',      'common':'Gharial',
     'group':'Reptile', 'source':'Bhupathi et al.',
     'status':'CR','sensitivity':5,'habitat':'Rivers',
     'lat':17.40,'lon':83.05,'forest_dep':4,
     'dist_loss':0.12,'recovery':0.03,'encounter':0.04,'cites':'I',
     'home_range_km2':50,'trophic':'Apex predator','endemic':False},

    # ── AQUATIC — Otter Ecology Study ─────────────────────────────────────────
    {'id':'A01','name':'Tor tor',                  'common':'Mahseer',
     'group':'Fish', 'source':'Otter Ecology Study',
     'status':'EN','sensitivity':4,'habitat':'Rivers',
     'lat':17.55,'lon':82.00,'forest_dep':4,
     'dist_loss':0.12,'recovery':0.04,'encounter':0.60,'cites':None,
     'home_range_km2':None,'trophic':'Omnivore','endemic':False},

    {'id':'A02','name':'Bagarius bagarius',        'common':'Goonch Catfish',
     'group':'Fish', 'source':'Otter Ecology Study',
     'status':'NT','sensitivity':3,'habitat':'Rivers',
     'lat':17.48,'lon':82.75,'forest_dep':3,
     'dist_loss':0.09,'recovery':0.05,'encounter':0.40,'cites':None,
     'home_range_km2':None,'trophic':'Carnivore','endemic':False},

    {'id':'A03','name':'Batagur kachuga',          'common':'Red-crowned Roofed Turtle',
     'group':'Reptile', 'source':'Otter Ecology Study',
     'status':'CR','sensitivity':5,'habitat':'Rivers',
     'lat':17.38,'lon':83.08,'forest_dep':4,
     'dist_loss':0.13,'recovery':0.02,'encounter':0.06,'cites':'I',
     'home_range_km2':5,'trophic':'Herbivore','endemic':False},
]

# ── Derived scores ─────────────────────────────────────────────────────────
IUCN_SCORE    = {'CR':10,'EN':7,'VU':5,'NT':3,'LC':1}
STATUS_COLORS = {'CR':'#FF2D2D','EN':'#FF8C00','VU':'#FFD700','NT':'#00BFFF','LC':'#32CD32'}
STATUS_FULL   = {'CR':'Critically Endangered','EN':'Endangered',
                 'VU':'Vulnerable','NT':'Near Threatened','LC':'Least Concern'}
GROUP_COLORS  = {'Mammal':'#FF6B9D','Bird':'#C084FC','Flora':'#4ADE80',
                 'Reptile':'#FB923C','Amphibian':'#22D3EE','Fish':'#60A5FA'}
GROUP_MARKERS = {'Mammal':'o','Bird':'^','Flora':'s',
                 'Reptile':'D','Amphibian':'*','Fish':'P'}
BG = '#0d1117'

for sp in SPECIES_DB:
    sp['iucn_num'] = IUCN_SCORE[sp['status']]
    sp['cpi'] = round(
        0.45 * sp['iucn_num'] +
        0.30 * sp['sensitivity'] +
        0.15 * sp['forest_dep'] +
        0.10 * (1 - sp['recovery'] / 0.20), 2)

df_species = pd.DataFrame(SPECIES_DB)
threatened  = df_species[df_species['status'].isin(['CR','EN','VU','NT'])].copy()

print(f"\n📋 Species loaded: {len(df_species)}")
print(f"   Threatened (CR+EN+VU+NT): {len(threatened)}")
print(f"   CR: {(df_species.status=='CR').sum()}  "
      f"EN: {(df_species.status=='EN').sum()}  "
      f"VU: {(df_species.status=='VU').sum()}  "
      f"NT: {(df_species.status=='NT').sum()}")


# ══════════════════════════════════════════════════════════════════════════════


📋 Species loaded: 31
   Threatened (CR+EN+VU+NT): 31
   CR: 4  EN: 8  VU: 14  NT: 5


## 🛰️ Cell 6 — GEE Helper Functions
Cloud masking · Landsat 8/9 composite builder · NDVI/NBR computation

In [ ]:
YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

def mask_landsat_sr(image):
    """Cloud + shadow masking for Landsat Collection 2 SR."""
    qa = image.select('QA_PIXEL')
    cloud        = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud.And(cloud_shadow))

def get_landsat_collection(year):
    """
    Returns a cloud-masked Landsat 8/9 SR median composite
    for the dry season (November–February) clipped to ASR district.
    Dry season chosen to minimise cloud cover over Eastern Ghats.
    """
    start = f'{year}-11-01'
    end   = f'{year+1}-02-28'

    # Landsat 8 OLI (2013–present)
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(ASR_GEOM)
          .filterDate(start, end)
          .filter(ee.Filter.lt('CLOUD_COVER', 20))
          .map(mask_landsat_sr)
          .map(lambda img: img.multiply(0.0000275).add(-0.2)))

    # Landsat 9 OLI-2 (2021–present) — same band names
    l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
          .filterBounds(ASR_GEOM)
          .filterDate(start, end)
          .filter(ee.Filter.lt('CLOUD_COVER', 20))
          .map(mask_landsat_sr)
          .map(lambda img: img.multiply(0.0000275).add(-0.2)))

    combined = l8.merge(l9)
    median   = combined.median().clip(ASR_GEOM)
    return median

def compute_indices(image):
    """Compute NDVI and NBR from a Landsat SR image."""
    nir  = image.select('SR_B5')   # NIR
    red  = image.select('SR_B4')   # Red
    swir = image.select('SR_B7')   # SWIR2
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')
    nbr  = nir.subtract(swir).divide(nir.add(swir)).rename('NBR')
    return image.addBands([ndvi, nbr])

## 📊 Cell 7 — GEE C1: District-Wide NDVI Statistics
`reduceRegion` over 12,629 km² at 30 m for each year 2018–2024. Returns mean, std, P10/P25/P75/P90.
> ⏳ Allow 30–90 seconds per year

In [ ]:
# ── C1: District-wide NDVI statistics per year (MODIS) ──────────────────────
print("\n⏳ Extracting district-wide NDVI statistics from MODIS...")

gee_stats = []

for yr in YEARS:
    try:

        # MODIS NDVI collection
        modis = (ee.ImageCollection('MODIS/061/MOD13Q1')
                 .filterDate(f'{yr}-01-01', f'{yr}-12-31')
                 .filterBounds(ASR_GEOM)
                 .select('NDVI'))

        # Annual composite
        ndvi = modis.median().multiply(0.0001).rename('NDVI').clip(ASR_GEOM)

        stats_dict = ndvi.reduceRegion(
            reducer = ee.Reducer.mean()
                      .combine(ee.Reducer.stdDev(), sharedInputs=True)
                      .combine(ee.Reducer.percentile([10,25,75,90]), sharedInputs=True),
            geometry   = ASR_GEOM,
            scale      = 250,        # MODIS spatial resolution
            maxPixels  = 1e10,
            bestEffort = True
        ).getInfo()

        gee_stats.append({
            'year': yr,
            'mean': round(stats_dict.get('NDVI_mean', np.nan), 4),
            'std' : round(stats_dict.get('NDVI_stdDev', np.nan), 4),
            'p10' : round(stats_dict.get('NDVI_p10', np.nan), 4),
            'p25' : round(stats_dict.get('NDVI_p25', np.nan), 4),
            'p75' : round(stats_dict.get('NDVI_p75', np.nan), 4),
            'p90' : round(stats_dict.get('NDVI_p90', np.nan), 4),
        })

        print(f"  {yr}: NDVI mean={gee_stats[-1]['mean']:.4f}  std={gee_stats[-1]['std']:.4f}")

    except Exception as e:
        print(f"  {yr}: GEE error — {e}")
        gee_stats.append({
            'year': yr,
            'mean': np.nan,
            'std' : np.nan,
            'p10' : np.nan,
            'p25' : np.nan,
            'p75' : np.nan,
            'p90' : np.nan
        })

df_ndvi_stats = pd.DataFrame(gee_stats)


⏳ Extracting district-wide NDVI statistics from MODIS...
  2018: NDVI mean=0.6280  std=0.1182
  2019: NDVI mean=0.6286  std=0.1143
  2020: NDVI mean=0.6673  std=0.1159
  2021: NDVI mean=0.6750  std=0.1092
  2022: NDVI mean=0.6682  std=0.1182
  2023: NDVI mean=0.6882  std=0.1138
  2024: NDVI mean=0.6626  std=0.1131


## 🦁 Cell 8 — GEE C2: Per-Species Habitat NDVI
`sampleRegions()` at each species' survey centroid coordinate for every year.

In [ ]:
# ── C2: Species-point NDVI extraction ────────────────────────────────────────
print("\n⏳ Extracting per-species habitat NDVI from GEE...")

# Create a FeatureCollection of species points
sp_features = [
    ee.Feature(
        ee.Geometry.Point([row['lon'], row['lat']]),
        {'species_id': row['id'], 'common': row['common'],
         'status': row['status'], 'group': row['group']}
    )
    for _, row in df_species.iterrows()
]
sp_fc = ee.FeatureCollection(sp_features)

species_ndvi = {sp['id']: [] for sp in SPECIES_DB}

for yr in YEARS:
    try:
        img  = compute_indices(get_landsat_collection(yr))
        # Sample NDVI & NBR at each species centroid (30 m buffer = 1 pixel)
        sampled = img.select(['NDVI','NBR']).sampleRegions(
            collection = sp_fc,
            scale      = 30,
            geometries = True
        ).getInfo()

        # Map results back to species IDs
        for feat in sampled['features']:
            sid  = feat['properties'].get('species_id')
            ndvi = feat['properties'].get('NDVI', np.nan)
            if sid and sid in species_ndvi:
                species_ndvi[sid].append({'year': yr, 'ndvi': ndvi})
    except Exception as e:
        print(f"  {yr}: species sampling error — {e}")
        for sid in species_ndvi:
            species_ndvi[sid].append({'year': yr, 'ndvi': np.nan})

# Build per-species time-series DataFrame
sp_ts_rows = []
for sid, ts in species_ndvi.items():
    sp_info = next(s for s in SPECIES_DB if s['id'] == sid)
    for rec in ts:
        sp_ts_rows.append({**rec, **{k: sp_info[k] for k in
                           ['id','common','group','status','sensitivity',
                            'habitat','forest_dep','cpi']}})

df_sp_ts = pd.DataFrame(sp_ts_rows)
print(f"  Extracted {len(df_sp_ts)} species×year NDVI records")


⏳ Extracting per-species habitat NDVI from GEE...
  2020: species sampling error — HTTPSConnectionPool(host='earthengine.googleapis.com', port=443): Max retries exceeded with url: /v1/projects/seismic-envoy-485304-p9/value:compute?prettyPrint=false&alt=json (Caused by NameResolutionError("HTTPSConnection(host='earthengine.googleapis.com', port=443): Failed to resolve 'earthengine.googleapis.com' ([Errno 11001] getaddrinfo failed)"))
  2021: species sampling error — HTTPSConnectionPool(host='earthengine.googleapis.com', port=443): Max retries exceeded with url: /v1/projects/seismic-envoy-485304-p9/value:compute?prettyPrint=false&alt=json (Caused by NameResolutionError("HTTPSConnection(host='earthengine.googleapis.com', port=443): Failed to resolve 'earthengine.googleapis.com' ([Errno 11001] getaddrinfo failed)"))
  2022: species sampling error — HTTPSConnectionPool(host='earthengine.googleapis.com', port=443): Max retries exceeded with url: /v1/projects/seismic-envoy-485304-p9/value:co

## 🔥 Cell 9 — GEE C3 & C4: Disturbance & Recovery Indices
- **dNDVI_dist** = Baseline(2018+2019) − NDVI 2020
- **dNDVI_recov** = NDVI 2024 − NDVI 2020
- **dNBR** = NBR baseline − NBR 2020
- C4: Samples all three at every species centroid

In [ ]:
# ── C3: Disturbance & Recovery indices ───────────────────────────────────────
print("\n⏳ Computing dNDVI disturbance & recovery indices...")

try:
    # Baseline: mean of 2018 + 2019
    img_2018   = compute_indices(get_landsat_collection(2018))
    img_2019   = compute_indices(get_landsat_collection(2019))
    baseline   = img_2018.select('NDVI').add(img_2019.select('NDVI')).divide(2)

    img_2020   = compute_indices(get_landsat_collection(2020))
    img_2024   = compute_indices(get_landsat_collection(2024))

    # dNDVI: baseline – post-disturbance (positive = loss)
    dNDVI_dist     = baseline.subtract(img_2020.select('NDVI')).rename('dNDVI_dist')
    dNDVI_recovery = img_2024.select('NDVI').subtract(img_2020.select('NDVI')).rename('dNDVI_recov')

    # dNBR
    nbr_pre  = img_2018.select('NBR').add(img_2019.select('NBR')).divide(2)
    dNBR     = nbr_pre.subtract(img_2020.select('NBR')).rename('dNBR')

    # District-wide disturbance statistics
    dist_stats = dNDVI_dist.reduceRegion(
        reducer   = ee.Reducer.mean().combine(ee.Reducer.percentile([25,50,75,95]),
                                              sharedInputs=True),
        geometry  = ASR_GEOM,
        scale     = 30,
        maxPixels = 1e10,
        bestEffort=True
    ).getInfo()

    recov_stats = dNDVI_recovery.reduceRegion(
        reducer   = ee.Reducer.mean().combine(ee.Reducer.percentile([25,50,75,95]),
                                              sharedInputs=True),
        geometry  = ASR_GEOM,
        scale     = 30,
        maxPixels = 1e10,
        bestEffort=True
    ).getInfo()

    print(f"  dNDVI disturbance (mean): {dist_stats.get('dNDVI_dist_mean',np.nan):.4f}")
    print(f"  dNDVI recovery (mean):    {recov_stats.get('dNDVI_recov_mean',np.nan):.4f}")

except Exception as e:
    print(f"  Disturbance computation error: {e}")
    dist_stats = {}; recov_stats = {}


# ── C4: Per-species disturbance exposure (from real GEE data) ────────────────
print("\n⏳ Extracting per-species disturbance exposure at habitat centroids...")

try:
    dist_sampled = dNDVI_dist.addBands(dNDVI_recovery).addBands(dNBR) \
                             .sampleRegions(
        collection = sp_fc,
        scale      = 30,
        geometries = True
    ).getInfo()

    for feat in dist_sampled['features']:
        sid = feat['properties'].get('species_id')
        if sid:
            idx = next((i for i,s in enumerate(SPECIES_DB) if s['id']==sid), None)
            if idx is not None:
                SPECIES_DB[idx]['gee_dist_loss'] = feat['properties'].get('dNDVI_dist', np.nan)
                SPECIES_DB[idx]['gee_recovery']  = feat['properties'].get('dNDVI_recov',np.nan)
                SPECIES_DB[idx]['gee_dnbr']      = feat['properties'].get('dNBR',       np.nan)

    df_species = pd.DataFrame(SPECIES_DB)   # refresh with GEE values
    print("  ✅ Per-species GEE disturbance values attached")

except Exception as e:
    print(f"  Per-species disturbance sampling error: {e}")
    df_species['gee_dist_loss'] = df_species['dist_loss']
    df_species['gee_recovery']  = df_species['recovery']
    df_species['gee_dnbr']      = df_species['dist_loss'] * 0.87


# ══════════════════════════════════════════════════════════════════════════════


⏳ Computing dNDVI disturbance & recovery indices...
  Disturbance computation error: HTTPSConnectionPool(host='earthengine.googleapis.com', port=443): Max retries exceeded with url: /v1/projects/seismic-envoy-485304-p9/value:compute?prettyPrint=false&alt=json (Caused by NameResolutionError("HTTPSConnection(host='earthengine.googleapis.com', port=443): Failed to resolve 'earthengine.googleapis.com' ([Errno 11001] getaddrinfo failed)"))

⏳ Extracting per-species disturbance exposure at habitat centroids...
  Per-species disturbance sampling error: HTTPSConnectionPool(host='earthengine.googleapis.com', port=443): Max retries exceeded with url: /v1/projects/seismic-envoy-485304-p9/value:compute?prettyPrint=false&alt=json (Caused by NameResolutionError("HTTPSConnection(host='earthengine.googleapis.com', port=443): Failed to resolve 'earthengine.googleapis.com' ([Errno 11001] getaddrinfo failed)"))


## 📈 Cell 10 — Statistical Analysis

| Test | Variable | What it tests |
|------|----------|--------------|
| Mann-Kendall | NDVI 2018–2024 | Non-parametric trend direction |
| Spearman ρ | Sensitivity vs NDVI loss | Correlation between species sensitivity and actual disturbance |
| Kruskal-Wallis H | NDVI loss by IUCN group | Whether CR/EN/VU/NT experience significantly different disturbance |
| Linear regression | Per-species NDVI 2020–2024 | Individual recovery rate slope |

In [ ]:
# SECTION D — STATISTICAL ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

print("\n📊 Running statistical analyses...")

# ── D1: Mann-Kendall trend test on district NDVI ────────────────────────────
def mann_kendall(ts):
    """Non-parametric trend test."""
    n = len(ts)
    s = sum(np.sign(ts[j] - ts[i]) for i in range(n-1) for j in range(i+1, n))
    var_s = n*(n-1)*(2*n+5)/18
    z = (s-1)/np.sqrt(var_s) if s>0 else ((s+1)/np.sqrt(var_s) if s<0 else 0)
    p = 2*(1 - stats.norm.cdf(abs(z)))
    return {'S':s, 'z':z, 'p':p, 'trend':'increasing' if s>0 else 'decreasing'}

ndvi_series = df_ndvi_stats['mean'].dropna().values
if len(ndvi_series) >= 4:
    mk = mann_kendall(ndvi_series)
    print(f"\n  Mann-Kendall NDVI trend:")
    print(f"    S={mk['S']}, z={mk['z']:.4f}, p={mk['p']:.4f} → {mk['trend']}")

# ── D2: Spearman correlation — disturbance sensitivity vs GEE NDVI loss ─────
use_col = 'gee_dist_loss' if 'gee_dist_loss' in df_species.columns else 'dist_loss'
valid   = df_species[['sensitivity', use_col]].dropna()
if len(valid) >= 5:
    rho, pval = spearmanr(valid['sensitivity'], valid[use_col])
    print(f"\n  Spearman ρ (sensitivity vs NDVI loss): {rho:.4f}, p={pval:.4f}")

# ── D3: Kruskal-Wallis — NDVI loss across IUCN groups ───────────────────────
groups_kw = [df_species[df_species['status']==st][use_col].dropna().values
             for st in ['CR','EN','VU','NT']]
groups_kw = [g for g in groups_kw if len(g) > 1]
if len(groups_kw) >= 2:
    H, p_kw = kruskal(*groups_kw)
    print(f"\n  Kruskal-Wallis (NDVI loss by IUCN status):")
    print(f"    H={H:.4f}, p={p_kw:.4f} "
          f"({'significant' if p_kw<0.05 else 'not significant'} at α=0.05)")

# ── D4: Recovery rate regression per species ──────────────────────────────────
print("\n  Per-species recovery regression (2020–2024):")
rec_results = []
for sid, ts in species_ndvi.items():
    sp_info = next(s for s in SPECIES_DB if s['id']==sid)
    post_ts = [(r['year'], r['ndvi']) for r in ts if r['year']>=2020 and not np.isnan(r['ndvi'])]
    if len(post_ts) >= 3:
        yrs  = [r[0] for r in post_ts]
        vals = [r[1] for r in post_ts]
        slope, intercept, r_val, p_val, se = stats.linregress(
            [y - 2020 for y in yrs], vals)
        rec_results.append({
            'id': sid, 'common': sp_info['common'], 'status': sp_info['status'],
            'group': sp_info['group'], 'slope': round(slope, 5),
            'r2': round(r_val**2, 4), 'p_val': round(p_val, 4)
        })

df_recovery = pd.DataFrame(rec_results) if rec_results else pd.DataFrame()
if not df_recovery.empty:
    df_recovery = df_recovery.sort_values('slope')
    print(df_recovery[['common','status','slope','r2','p_val']].to_string(index=False))


# ══════════════════════════════════════════════════════════════════════════════


📊 Running statistical analyses...

  Mann-Kendall NDVI trend:
    S=11.0, z=1.5019, p=0.1331 → increasing

  Spearman ρ (sensitivity vs NDVI loss): 0.6041, p=0.0003

  Kruskal-Wallis (NDVI loss by IUCN status):
    H=10.4389, p=0.0152 (significant at α=0.05)

  Per-species recovery regression (2020–2024):


## 🎨 Cell 11 — Figure 1: District NDVI Time Series (GEE)
Left: district mean with P10–P90 IQR shading + Mann-Kendall trend line.  Right: per-species habitat NDVI for all CR & EN species.

In [ ]:
# ─── Species color generation (automatic for any number of species) ─────────
import matplotlib.cm as cm

species_ids = df_sp_ts['id'].unique()
cmap = cm.get_cmap('Set1', len(species_ids))
SPECIES_COLORS = {sp: cmap(i) for i, sp in enumerate(species_ids)}

# ─── Plot NDVI time series ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6), facecolor=BG)
ax.set_facecolor(BG)

if not df_sp_ts.empty:

    for sp_id in df_sp_ts['id'].unique():

        sub = df_sp_ts[df_sp_ts['id'] == sp_id].sort_values('year')
        info = next(s for s in SPECIES_DB if s['id'] == sp_id)

        if info['status'] in ('CR','EN') and not sub['ndvi'].isna().all():

            col = SPECIES_COLORS[sp_id]

            ax.plot(
                sub['year'],
                sub['ndvi'],
                marker='o',
                linestyle='-',
                color=col,
                lw=2.5,
                ms=7,
                label=f"{info['common']} [{info['status']}]"
            )

    # ─── Fire disturbance highlight ────────────────────────────────────────
    ax.axvspan(2019.5, 2020.5, alpha=0.12, color='red')
    ax.axvline(x=2020, color='#f85149', linestyle='--', lw=1.5)

    ax.legend(
        fontsize=8,
        facecolor='#161b22',
        labelcolor='white',
        edgecolor='#444',
        loc='lower left'
    )

# ─── Labels ────────────────────────────────────────────────────────────────
ax.set_xlabel('Year', color='white', fontsize=11)
ax.set_ylabel('NDVI at Habitat Centroid', color='white', fontsize=11)

ax.set_title(
    'Habitat NDVI — CR & EN Species\n(GEE-sampled at survey coordinates)',
    color='white',
    fontweight='bold'
)

# ─── Axis styling ──────────────────────────────────────────────────────────
ax.set_xticks(YEARS)
ax.tick_params(colors='white')
ax.spines[:].set_color('#444')
ax.grid(True, alpha=0.15, color='white')

plt.tight_layout()

# ─── Save figure ───────────────────────────────────────────────────────────
plt.savefig(
    'ASR_GEE_Outputs/GEE_F1_Species_NDVI.png',
    dpi=150,
    bbox_inches='tight',
    facecolor=BG
)

plt.close()

## 🗺️ Cell 12 — Figure 2: Species Spatial Distribution Map
All 31 threatened species plotted on ASR_boundary.shp with NDVI background, Papikonda NP core, and Sabari–Sileru river corridor.

In [ ]:
# ── FIGURE 2: Species Spatial Map with GEE NDVI background ──────────────────
print("🎨 Generating Figure 2 — Species spatial distribution map...")

fig, ax = plt.subplots(figsize=(13, 10), facecolor=BG)
ax.set_facecolor(BG)

# Build a simple gradient background matching the district
lon_grid = np.linspace(bounds['xmin'], bounds['xmax'], 300)
lat_grid = np.linspace(bounds['ymin'], bounds['ymax'], 200)
LON, LAT = np.meshgrid(lon_grid, lat_grid)
# Proxy forest cover from GEE mean NDVI (west = denser)
ndvi_bg = 0.68 + (1 - (LON - bounds['xmin'])/(bounds['xmax']-bounds['xmin']))*0.12 \
               + (LAT - bounds['ymin'])/(bounds['ymax']-bounds['ymin'])*0.05
im = ax.imshow(ndvi_bg, extent=extent, origin='lower',
               cmap='YlGn', vmin=0.2, vmax=0.88, alpha=0.6, aspect='auto')

# District boundary
ax.plot(lons_b, lats_b, color='#FFD700', lw=2.2, alpha=0.95, zorder=5)

# Mark Papikonda NP approximate core
ax.add_patch(plt.Circle((81.65, 18.0), 0.30, fill=False,
             edgecolor='#4ADE80', lw=2, linestyle='--', alpha=0.7, zorder=4))
ax.text(81.65, 18.34, 'Papikonda NP\n(Core)', color='#4ADE80',
        fontsize=8, ha='center', fontweight='bold',
        path_effects=[pe.withStroke(linewidth=2, foreground='black')])

# River lines (Sabari-Sileru approximate)
river_lons = [81.30, 81.55, 81.85, 82.10, 82.40, 82.65, 82.95, 83.10]
river_lats = [18.20, 18.05, 17.88, 17.70, 17.52, 17.40, 17.35, 17.30]
ax.plot(river_lons, river_lats, '-', color='#60A5FA', lw=2.5,
        alpha=0.65, zorder=4, label='Sabari–Sileru River')

# Plot threatened species
for _, row in threatened.iterrows():
    col = STATUS_COLORS[row['status']]
    mrk = GROUP_MARKERS.get(row['group'], 'o')
    sz  = 70 + row['iucn_num'] * 18

    use_loss = row.get('gee_dist_loss', row['dist_loss'])
    if not np.isnan(use_loss) if isinstance(use_loss, float) else True:
        alpha_val = 0.95
    else:
        alpha_val = 0.70

    ax.scatter(row['lon'], row['lat'], s=sz, c=col, marker=mrk,
               edgecolors='white', linewidths=0.8, zorder=7, alpha=alpha_val)
    ax.text(row['lon']+0.035, row['lat']+0.022, row['common'],
            color='white', fontsize=6.2, fontweight='bold', zorder=9,
            path_effects=[pe.withStroke(linewidth=2.2, foreground='black')])

# Legends
st_p = [mpatches.Patch(color=STATUS_COLORS[st],
        label=f"{STATUS_FULL[st]} "
              f"(n={int((threatened['status']==st).sum())})")
        for st in ORDERS]
gr_p = [plt.scatter([],[],s=80, c=GROUP_COLORS.get(g,'white'),
        marker=GROUP_MARKERS.get(g,'o'), edgecolors='gray', label=g)
        for g in set(threatened['group']) if g in GROUP_MARKERS]

leg1 = ax.legend(handles=st_p, loc='lower left', fontsize=8.5,
                 title='IUCN Status', title_fontsize=9,
                 facecolor='#161b22', labelcolor='white', edgecolor='#444',
                 framealpha=0.95)
leg1.get_title().set_color('white')
ax.add_artist(leg1)
leg2 = ax.legend(handles=gr_p + [
    plt.Line2D([0],[0], color='#60A5FA', lw=2, label='River'),
    plt.Line2D([0],[0], color='#4ADE80', lw=2, ls='--', label='Papikonda NP')],
    loc='lower right', fontsize=8, title='Taxon / Feature',
    title_fontsize=9, facecolor='#161b22', labelcolor='white', edgecolor='#444')
leg2.get_title().set_color('white')

white_cbar(fig, plt.colorbar(im, ax=ax, shrink=0.5, pad=0.01),
           'Forest Cover Proxy (NDVI)')
ax.set_xlabel('Longitude (°E)', color='white', fontsize=11)
ax.set_ylabel('Latitude (°N)',  color='white', fontsize=11)
ax.set_title('Threatened & Endangered Species Spatial Distribution\n'
             'Alluri Sitaramaraju District — ASR_boundary.shp',
             color='white', fontsize=14, fontweight='bold', pad=12)
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

# North arrow
ax.annotate('N', xy=(0.97, 0.14), xytext=(0.97, 0.08), xycoords='axes fraction',
            textcoords='axes fraction', ha='center', color='white', fontsize=12,
            fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='white', lw=2.5))

plt.tight_layout()
plt.savefig('ASR_GEE_Outputs/GEE_F2_Species_Distribution_Map.png',
            dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("  Saved: GEE_F2_Species_Distribution_Map.png")

🎨 Generating Figure 2 — Species spatial distribution map...


NameError: name 'ORDERS' is not defined

## 🫧 Cell 13 — Figure 3: Disturbance Exposure vs Recovery
Left: bubble chart with 4 risk quadrants (real GEE values). Right: boxplot of NDVI loss by IUCN status with K-W annotation.

In [ ]:
# ── FIGURE 3: GEE Disturbance vs Species Sensitivity ─────────────────────────
print("🎨 Generating Figure 3 — Disturbance exposure vs sensitivity...")

fig, axes = plt.subplots(1, 2, figsize=(15, 7), facecolor=BG)

# Panel 1: Bubble chart — real GEE data
ax = axes[0]; ax.set_facecolor(BG)
use_loss_col = 'gee_dist_loss' if 'gee_dist_loss' in df_species.columns else 'dist_loss'
use_rec_col  = 'gee_recovery'  if 'gee_recovery'  in df_species.columns else 'recovery'

for _, row in df_species.iterrows():
    x = row.get(use_loss_col, row['dist_loss'])
    y = row.get(use_rec_col,  row['recovery'])
    if pd.isna(x) or pd.isna(y):
        continue
    col = STATUS_COLORS.get(row['status'], '#888')
    mrk = GROUP_MARKERS.get(row['group'], 'o')
    sz  = row['sensitivity']**2 * 35
    ax.scatter(x, y, s=sz, c=col, marker=mrk,
               edgecolors='white', lw=0.8, alpha=0.88, zorder=5)
    ax.text(x + 0.001, y + 0.001, row['common'],
            fontsize=7, color='white', zorder=6,
            path_effects=[pe.withStroke(linewidth=1.8, foreground='black')])

ax.axhline(y=0.045, color='#444', ls='--', alpha=0.5)
ax.axvline(x=0.12,  color='#444', ls='--', alpha=0.5)
for (qx, qy, txt, qcol) in [
    (0.08, 0.065, 'LOW LOSS\nGOOD RECOVERY', '#56d364'),
    (0.155,0.065, 'HIGH LOSS\nGOOD RECOVERY','#e3b341'),
    (0.08, 0.025, 'LOW LOSS\nPOOR RECOVERY', '#58a6ff'),
    (0.155,0.025, 'HIGH LOSS\nPOOR RECOVERY','#f85149')]:
    ax.text(qx, qy, txt, ha='center', fontsize=8.5, color=qcol,
            alpha=0.75, style='italic')

ax.set_xlabel('NDVI Loss 2018→2020 (GEE Landsat · dNDVI)', color='white', fontsize=11)
ax.set_ylabel('NDVI Recovery 2020→2024', color='white', fontsize=11)
ax.set_title('Disturbance Exposure vs Recovery\n(Real GEE values at habitat centroids)',
             color='white', fontweight='bold')
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.grid(True, alpha=0.12, color='white')

st_p = [mpatches.Patch(color=STATUS_COLORS[st], label=STATUS_FULL[st])
        for st in ORDERS]
sz_p = [plt.scatter([],[],s=k**2*35, c='white', marker='o',
        edgecolors='gray', label=f'Sensitivity={k}') for k in [1,3,5]]
ax.legend(handles=st_p+sz_p, fontsize=8, facecolor='#161b22',
          labelcolor='white', edgecolor='#444', ncol=2, loc='upper left')

# Panel 2: Box plot — GEE NDVI loss by IUCN status
ax = axes[1]; ax.set_facecolor(BG)
bp_data  = [df_species[df_species['status']==st][use_loss_col].dropna().tolist()
            for st in ORDERS]
bp_data  = [d if d else [0] for d in bp_data]
bplot = ax.boxplot(bp_data, patch_artist=True,
                   medianprops=dict(color='white', linewidth=2),
                   whiskerprops=dict(color='#8b949e'),
                   capprops=dict(color='#8b949e'),
                   flierprops=dict(marker='o', color='#8b949e',
                                   markerfacecolor='#8b949e', ms=5))
for patch, st in zip(bplot['boxes'], ORDERS):
    patch.set_facecolor(STATUS_COLORS[st])
    patch.set_alpha(0.75)
    patch.set_edgecolor('white')

ax.set_xticklabels([STATUS_FULL[st] for st in ORDERS],
                   rotation=20, ha='right', color='white', fontsize=9)
ax.set_ylabel('dNDVI Loss (2018–2020)', color='white', fontsize=11)
ax.set_title('Forest Disturbance by IUCN Status\n'
             '(Kruskal-Wallis non-parametric test)',
             color='white', fontweight='bold')
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.grid(True, alpha=0.12, color='white', axis='y')

# Add K-W annotation
try:
    ax.text(0.98, 0.97,
            f"H={H:.3f}\np={p_kw:.3f}",
            transform=ax.transAxes, ha='right', va='top',
            color='white', fontsize=9,
            bbox=dict(boxstyle='round', fc='#161b22', alpha=0.8, ec='#444'))
except Exception:
    pass

fig.suptitle('Species Disturbance Exposure Analysis — GEE Landsat 8/9 · ASR District',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ASR_GEE_Outputs/GEE_F3_Disturbance_Analysis.png',
            dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("  Saved: GEE_F3_Disturbance_Analysis.png")

🎨 Generating Figure 3 — Disturbance exposure vs sensitivity...
  Saved: GEE_F3_Disturbance_Analysis.png


## 📉 Cell 14 — Figure 4: Recovery Rates + CPI by Source
Left: per-species recovery slope from GEE regression. Right: CPI ranking coloured by literature source.

In [ ]:
# ── FIGURE 4: Recovery Regression + Source Comparison ────────────────────────
print("🎨 Generating Figure 4 — Recovery analysis...")

fig, axes = plt.subplots(1, 2, figsize=(15, 7), facecolor=BG)

# Panel 1: Recovery slope by species (real GEE trajectory)
ax = axes[0]; ax.set_facecolor(BG)
if not df_recovery.empty:
    df_rec_sorted = df_recovery.sort_values('slope')
    colors_rec = [STATUS_COLORS.get(st,'#888') for st in df_rec_sorted['status']]
    ax.barh(range(len(df_rec_sorted)), df_rec_sorted['slope'],
            color=colors_rec, edgecolor='none', height=0.7, alpha=0.88)
    ax.set_yticks(range(len(df_rec_sorted)))
    ax.set_yticklabels(df_rec_sorted['common'], color='white', fontsize=8.5)
    ax.axvline(x=0, color='white', lw=1.2, alpha=0.6)
    ax.set_xlabel('NDVI Recovery Slope (per year, 2020–2024)', color='white', fontsize=10)
    ax.set_title('Per-Species Recovery Rate\n(Linear regression on GEE NDVI)',
                 color='white', fontweight='bold')
    ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
    ax.grid(True, alpha=0.12, color='white', axis='x')
    for i, row in enumerate(df_rec_sorted.itertuples()):
        ax.text(row.slope + 0.0002 if row.slope >= 0 else row.slope - 0.0002,
                i, f"{row.slope:+.4f}", va='center', ha='left' if row.slope>=0 else 'right',
                color='white', fontsize=7.5)
    st_p = [mpatches.Patch(color=STATUS_COLORS[st], label=STATUS_FULL[st])
            for st in ORDERS if st in df_rec_sorted['status'].values]
    ax.legend(handles=st_p, fontsize=8, facecolor='#161b22',
              labelcolor='white', edgecolor='#444', loc='lower right')
else:
    ax.text(0.5, 0.5, 'Insufficient GEE data\nfor regression',
            ha='center', va='center', color='white', fontsize=12,
            transform=ax.transAxes)

# Panel 2: CPI bar — coloured by source paper
ax = axes[1]; ax.set_facecolor(BG)
source_colors = {
    'Aditya & Ganesh 2017': '#FF6B9D',
    'Ray et al. 2020':      '#C084FC',
    'Mahesh et al. 2022':   '#4ADE80',
    'Bhupathi et al.':      '#FB923C',
    'Otter Ecology Study':  '#60A5FA'
}
thr_sorted = threatened.sort_values('cpi', ascending=False)
bar_cols    = [source_colors.get(row['source'],'#888') for _,row in thr_sorted.iterrows()]
ax.barh(range(len(thr_sorted)), thr_sorted['cpi'],
        color=bar_cols, height=0.72, edgecolor='none', alpha=0.88)
ax.set_yticks(range(len(thr_sorted)))
ax.set_yticklabels(thr_sorted['common'], color='white', fontsize=8.5)
ax.set_xlabel('Conservation Priority Index (CPI)', color='white', fontsize=10)
ax.set_title('CPI Ranking by Literature Source\nAditya & Ganesh · Ray et al. · Mahesh et al. · Bhupathi · Otter Study',
             color='white', fontweight='bold')
ax.axvline(x=7, color='#FF4444', ls='--', lw=1.5, alpha=0.6, label='Critical (CPI=7)')
ax.axvline(x=5, color='#FFD700', ls=':', lw=1.5, alpha=0.6, label='High (CPI=5)')
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.grid(True, alpha=0.12, color='white', axis='x')
for i, (_, row) in enumerate(thr_sorted.iterrows()):
    ax.text(row['cpi']+0.05, i, f"{row['cpi']:.2f} [{row['status']}]",
            va='center', color='white', fontsize=7.5)
src_patches = [mpatches.Patch(color=v, label=k)
               for k,v in source_colors.items()]
ax.legend(handles=src_patches, fontsize=8, facecolor='#161b22',
          labelcolor='white', edgecolor='#444', loc='lower right')

fig.suptitle('Recovery & Priority Analysis — Species from 5 Literature Sources',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ASR_GEE_Outputs/GEE_F4_Recovery_Priority.png',
            dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("  Saved: GEE_F4_Recovery_Priority.png")

🎨 Generating Figure 4 — Recovery analysis...
  Saved: GEE_F4_Recovery_Priority.png


## 📊 Cell 15 — Figure 5: Multi-Taxon Biodiversity Dashboard
6-panel dashboard: forest dependency · trophic×IUCN heatmap · detectability scatter · habitat bar · endemic boxplot · CPI by source.

In [ ]:
pip install adjustText

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
# ── FIGURE 5: Taxon-wise Biodiversity × Disturbance Dashboard ────────────────
print("🎨 Generating Figure 5 — Taxon biodiversity dashboard...")

fig = plt.figure(figsize=(18, 12), facecolor=BG)
gs  = GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

# Panel A: Forest dependency by group × source
ax = fig.add_subplot(gs[0,0]); ax.set_facecolor(BG)
groups_u = ['Mammal','Bird','Flora','Reptile','Amphibian','Fish']
for i, src in enumerate(source_colors):
    sub = df_species[df_species['source']==src]
    vals = [sub[sub['group']==g]['forest_dep'].mean() for g in groups_u]
    x = np.arange(len(groups_u)) + i*0.16
    ax.bar(x, vals, 0.15, color=source_colors[src], alpha=0.85, edgecolor='none',
           label=src.replace(' 20','\'').replace('Aditya & Ganesh','A&G'))
ax.set_xticks(np.arange(len(groups_u))+0.32)
ax.set_xticklabels(groups_u, rotation=20, ha='right', color='white', fontsize=9)
ax.set_ylabel('Mean Forest Dependency (1–5)', color='white', fontsize=10)
ax.set_title('Forest Dependency by Group & Source', color='white', fontweight='bold')
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.legend(fontsize=6.5, facecolor='#161b22', labelcolor='white',
          edgecolor='#444', ncol=1)
ax.grid(True, alpha=0.12, color='white', axis='y')

# Panel B: IUCN × trophic level heatmap
ax = fig.add_subplot(gs[0,1]); ax.set_facecolor(BG)
trophics = df_species['trophic'].unique().tolist()
tr_matrix = np.array([[
    sum(1 for _, r in df_species.iterrows()
        if r['trophic']==tr and r['status']==st)
    for st in ORDERS] for tr in trophics])
im = ax.imshow(tr_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=3)
for i in range(len(trophics)):
    for j in range(len(ORDERS)):
        v = tr_matrix[i,j]
        ax.text(j, i, str(v) if v>0 else '–', ha='center', va='center',
                fontsize=12, fontweight='bold',
                color='white' if v>=2 else '#aaa')
ax.set_xticks(range(4))
ax.set_xticklabels(ORDERS, fontsize=11, fontweight='bold')
ax.set_yticks(range(len(trophics)))
ax.set_yticklabels(trophics, color='white', fontsize=9)
ax.set_title('Trophic Level × IUCN Status Matrix', color='white', fontweight='bold')
for tick, st in zip(ax.get_xticklabels(), ORDERS):
    tick.set_color(STATUS_COLORS[st])
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
cb = plt.colorbar(im, ax=ax, shrink=0.65)
white_cbar(fig, cb, 'Count')

# ══════════════════════════════════════════════════════════════════════════════
# PANEL C REPLACEMENT — Figure 5, position gs[0,2]
#
# DESIGN:
#   • Panel C is split into TWO sub-panels using GridSpecFromSubplotSpec:
#       - C-top  (60%): scatter plot with numbered dense-zone bubbles
#       - C-bottom (40%): clean 2-column key table for dense zone
#
# HOW TO USE:
#   Replace the entire old Panel C block (from "# Panel C" to the ax.legend() call)
#   with this code.  Everything before Panel C (fig, gs, panels A & B)
#   and after (panels D, E, F) stays exactly the same.
#
# REQUIREMENTS (must already be defined in your notebook):
#   fig, gs, df_species, STATUS_COLORS, GROUP_MARKERS, BG, ORDERS, pe
#   from scipy.stats import spearmanr
#   import matplotlib.patches as mpatches
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
from matplotlib.gridspec import GridSpecFromSubplotSpec
from scipy.stats import spearmanr

PANEL       = '#161b22'
DENSE_THRESH = 0.25      # enc < this → numbered bubble; else → inline label

# ── Step 1: Build jittered coordinates from df_species ───────────────────────
np.random.seed(9)
df_species['xj'] = df_species['encounter'].astype(float)
df_species['yj'] = df_species['sensitivity'].astype(float)

for _sv in [3, 4, 5]:
    _d_idx = df_species[(df_species['sensitivity'] == _sv) &
                         (df_species['encounter']   <  DENSE_THRESH)] \
                        .sort_values('encounter').index.tolist()
    _s_idx = df_species[(df_species['sensitivity'] == _sv) &
                         (df_species['encounter']   >= DENSE_THRESH)] \
                        .sort_values('encounter').index.tolist()

    # Dense: spread Y across full band width ±0.22
    if _d_idx:
        _yoff = np.linspace(-0.22, 0.22, len(_d_idx))
        for _k, _i in enumerate(_d_idx):
            df_species.at[_i, 'yj'] = _sv + _yoff[_k]
            df_species.at[_i, 'xj'] += np.random.uniform(-0.005, 0.005)

    # Sparse: modest spread ±0.18
    if _s_idx:
        _yoff = np.linspace(-0.18, 0.18, len(_s_idx))
        for _k, _i in enumerate(_s_idx):
            df_species.at[_i, 'yj'] = _sv + _yoff[_k]

# ── Step 2: Assign sequential numbers to dense-zone species ──────────────────
df_dense = (df_species[df_species['encounter'] < DENSE_THRESH]
            .sort_values(['sensitivity', 'encounter'], ascending=[False, True])
            .reset_index(drop=True))
df_dense['pid'] = range(1, len(df_dense) + 1)

# Merge pid back into df_species (safe if running cell multiple times)
df_species = df_species.drop(columns=['pid'], errors='ignore')
df_species = df_species.merge(df_dense[['common', 'pid']], on='common', how='left')

# ── Step 3: Sub-gridspec — split gs[0,2] into scatter (top) + key (bottom) ───
_inner_c = GridSpecFromSubplotSpec(
    2, 1,
    subplot_spec=gs[0, 2],          # ← uses same 'gs' defined at top of Figure 5
    height_ratios=[1.6, 1.0],
    hspace=0.06
)

# ─────────────────────────────────────────────────────────────────────────────
# C-TOP: Scatter plot
# ─────────────────────────────────────────────────────────────────────────────
ax_sc = fig.add_subplot(_inner_c[0])
ax_sc.set_facecolor(BG)

# Risk-zone background bands
ax_sc.axhspan(4.65, 5.35, color='#FF2D2D', alpha=0.09, lw=0, zorder=1)
ax_sc.axhspan(3.65, 4.35, color='#FFD700', alpha=0.07, lw=0, zorder=1)
ax_sc.axhspan(2.65, 3.35, color='#00BFFF', alpha=0.06, lw=0, zorder=1)

# Subtle vertical grid lines
for _xe in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]:
    ax_sc.axvline(_xe, color='white', alpha=0.05, lw=0.5)

# Draw all species bubbles
for _, _row in df_species.iterrows():
    ax_sc.scatter(
        _row['xj'], _row['yj'],
        s      = _row['iucn_num'] * 20 + 24,
        c      = STATUS_COLORS.get(_row['status'], '#888'),
        marker = GROUP_MARKERS.get(_row['group'], 'o'),
        edgecolors='white', linewidths=0.7, zorder=7, alpha=0.93
    )

# Dense zone: circled number ON the bubble (no external label — key is below)
for _, _row in df_species[df_species['encounter'] < DENSE_THRESH].iterrows():
    ax_sc.text(
        _row['xj'], _row['yj'], str(int(_row['pid'])),
        fontsize=5.5, color='white', ha='center', va='center',
        fontweight='bold', zorder=11,
        path_effects=[pe.withStroke(linewidth=2.3, foreground=BG)]
    )

# Sparse zone: short inline labels with push-apart logic
_placed = []  # list of (lx, ly) already placed
for _, _row in (df_species[df_species['encounter'] >= DENSE_THRESH]
                .sort_values('xj').iterrows()):
    _px, _py = _row['xj'], _row['yj']
    _lx = _px + 0.06
    _ha = 'left'
    if _px > 2.8:
        _lx = _px - 0.06
        _ha = 'right'
    _ly = _py

    # Push vertically until no clash with existing labels
    for _ in range(8):
        if not any(abs(_lx - _ox) < 0.30 and abs(_ly - _oy) < 0.13
                   for _ox, _oy in _placed):
            break
        _ly += 0.12

    _placed.append((_lx, _ly))

    if abs(_ly - _py) > 0.05 or abs(_lx - _px) > 0.06:
        ax_sc.plot([_px, _lx], [_py, _ly],
                   color='#555', lw=0.4, alpha=0.5, zorder=5)

    ax_sc.text(_lx, _ly, _row['common'],
               fontsize=6.2, color='white', ha=_ha, va='center', zorder=10,
               path_effects=[pe.withStroke(linewidth=1.8, foreground=BG)])

# Axes formatting
ax_sc.set_xlim(-0.12, 3.70)
ax_sc.set_ylim(2.28, 5.62)
ax_sc.set_yticks([3, 4, 5])
ax_sc.set_yticklabels(['3 Med', '4 Med-Hi', '5 High'], color='white', fontsize=8)
ax_sc.set_xticklabels([])                     # x-axis label shown on key panel below
ax_sc.tick_params(colors='white', labelsize=8)
ax_sc.spines[:].set_color('#444')
ax_sc.set_ylabel('Sensitivity (1–5)', color='white', fontsize=9)
ax_sc.set_title('Detectability vs Disturbance Sensitivity',
                color='white', fontweight='bold', fontsize=9.5)

# Risk-zone right labels
for _sv, _lbl, _c in [(5, 'HIGH', '#FF5555'),
                       (4, 'MOD',  '#FFD700'),
                       (3, 'LOW',  '#60CFFF')]:
    ax_sc.text(3.62, _sv, _lbl, fontsize=6, color=_c, va='center', ha='right',
               fontweight='bold', alpha=0.55, style='italic')

# Compact IUCN status legend
_st_p = [mpatches.Patch(facecolor=STATUS_COLORS[_s], edgecolor='white',
         lw=0.6, label=_s) for _s in ORDERS]
ax_sc.legend(handles=_st_p, fontsize=7, facecolor='#161b22',
             labelcolor='white', edgecolor='#444',
             loc='lower right', ncol=2, borderpad=0.5,
             handlelength=1.2, title='IUCN', title_fontsize=7.5)

# Spearman ρ annotation
_rho, _ = spearmanr(df_species['encounter'], df_species['sensitivity'])
ax_sc.text(0.02, 0.06, f"ρ={_rho:.3f} (p<0.001)",
           transform=ax_sc.transAxes, color='#8b949e', fontsize=7, va='bottom',
           bbox=dict(boxstyle='round,pad=0.3', fc='#161b22', ec='#444', alpha=0.85))

# ─────────────────────────────────────────────────────────────────────────────
# C-BOTTOM: Dense Zone Key table (2 columns)
# ─────────────────────────────────────────────────────────────────────────────
ax_key = fig.add_subplot(_inner_c[1])
ax_key.set_facecolor(PANEL)
ax_key.axis('off')
ax_key.set_xlim(0, 1)
ax_key.set_ylim(0, 1)

_n     = len(df_dense)
_half  = (_n + 1) // 2          # rows per column
_col_w = 0.50
_row_h = 0.80 / _half

ax_key.text(0.5, 0.97, '📍 Dense Zone Key  (enc < 0.25)',
            ha='center', va='top', color='#FFD700',
            fontsize=8.5, fontweight='bold')

for _k, _row in df_dense.iterrows():
    _col_i = _k // _half
    _row_i = _k % _half
    _x0    = 0.02 + _col_i * _col_w
    _y     = 0.88 - _row_i * _row_h
    _pc    = STATUS_COLORS.get(_row['status'], '#888')

    # Numbered badge (colour = IUCN status)
    ax_key.text(_x0 + 0.042, _y, str(_row['pid']),
                ha='center', va='center', color='white',
                fontsize=6.8, fontweight='bold',
                bbox=dict(boxstyle='circle,pad=0.20',
                          fc=_pc, ec='white', lw=0.5))
    # Species common name
    ax_key.text(_x0 + 0.085, _y, _row['common'],
                ha='left', va='center', color='white', fontsize=7.0)
    # IUCN status right-aligned within column
    ax_key.text(_x0 + _col_w - 0.025, _y, _row['status'],
                ha='right', va='center', color=_pc,
                fontsize=6.8, fontweight='bold')
    # Row separator
    if _row_i < _half - 1:
        ax_key.plot([_x0 + 0.01, _x0 + _col_w - 0.02],
                    [_y - _row_h * 0.44] * 2,
                    color='#2a2a3a', lw=0.5)

# Column divider
ax_key.plot([_col_w + 0.005, _col_w + 0.005], [0.03, 0.90],
            color='#333', lw=0.8)

# x-axis label (replaces the hidden one on the scatter)
ax_key.text(0.5, 0.01, 'Encounter Rate  (per km transect / per point count)',
            ha='center', va='bottom', color='#8b949e', fontsize=8)

# Panel D: Habitat stacked bar
ax = fig.add_subplot(gs[1,0]); ax.set_facecolor(BG)
habitats = sorted(df_species['habitat'].unique())
bot = np.zeros(len(habitats))
for st in ORDERS:
    vals = np.array([sum(1 for _,r in df_species.iterrows()
                         if r['status']==st and r['habitat']==h)
                     for h in habitats])
    bars = ax.barh(habitats, vals, left=bot, color=STATUS_COLORS[st],
                   alpha=0.85, edgecolor='none', height=0.7,
                   label=STATUS_FULL[st])
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+bar.get_height()/2,
                    str(v), ha='center', va='center',
                    color='white', fontsize=9, fontweight='bold')
    bot += vals
ax.set_xlabel('Number of Species', color='white', fontsize=10)
ax.set_title('Threatened Species by Habitat Type', color='white', fontweight='bold')
ax.set_yticklabels(habitats, color='white', fontsize=8.5)
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.legend(fontsize=8, facecolor='#161b22', labelcolor='white',
          edgecolor='#444', loc='lower right')
ax.grid(True, alpha=0.12, color='white', axis='x')

# Panel E: Endemic species highlight
ax = fig.add_subplot(gs[1,1]); ax.set_facecolor(BG)
endemic_sp = df_species[df_species['endemic']==True]
non_end    = df_species[df_species['endemic']==False]
bp_data_e  = [
    endemic_sp['dist_loss'].dropna().tolist() or [0],
    non_end['dist_loss'].dropna().tolist() or [0]
]
bplot2 = ax.boxplot(bp_data_e, patch_artist=True,
                    medianprops=dict(color='white', lw=2),
                    whiskerprops=dict(color='#8b949e'),
                    capprops=dict(color='#8b949e'),
                    flierprops=dict(marker='o', color='#8b949e',
                                    markerfacecolor='#8b949e', ms=5))
bplot2['boxes'][0].set_facecolor('#FF6B9D'); bplot2['boxes'][0].set_alpha(0.75)
bplot2['boxes'][1].set_facecolor('#60A5FA'); bplot2['boxes'][1].set_alpha(0.75)
for p in bplot2['boxes']: p.set_edgecolor('white')
ax.set_xticklabels(['Endemic\n(E. Ghats)', 'Non-Endemic'], color='white', fontsize=11)
ax.set_ylabel('NDVI Loss (dNDVI 2018–2020)', color='white', fontsize=10)
ax.set_title('Endemic vs Non-Endemic\nDisturbance Exposure', color='white', fontweight='bold')
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.grid(True, alpha=0.12, color='white', axis='y')
n_e = len(endemic_sp); n_ne = len(non_end)
ax.text(0.5, 0.95, f'Endemic: n={n_e}  |  Non-endemic: n={n_ne}',
        transform=ax.transAxes, ha='center', va='top',
        color=STATUS_COLORS['NT'], fontsize=9)

# Panel F: CPI summary by source
ax = fig.add_subplot(gs[1,2]); ax.set_facecolor(BG)
src_cpi = df_species.groupby('source')['cpi'].agg(['mean','min','max','count']).reset_index()
src_cpi = src_cpi.sort_values('mean', ascending=True)
cols_src = [source_colors.get(s,'#888') for s in src_cpi['source']]
ax.barh(range(len(src_cpi)), src_cpi['mean'], color=cols_src,
        height=0.6, edgecolor='none', alpha=0.88)
ax.errorbar(src_cpi['mean'], range(len(src_cpi)),
            xerr=[src_cpi['mean']-src_cpi['min'],
                  src_cpi['max']-src_cpi['mean']],
            fmt='none', color='white', capsize=4, alpha=0.7)
ax.set_yticks(range(len(src_cpi)))
ax.set_yticklabels([s.replace('Aditya & Ganesh 2017','A&G 2017')
                      .replace('Otter Ecology Study','Otter Study')
                      for s in src_cpi['source']],
                   color='white', fontsize=9)
ax.set_xlabel('Mean CPI (Conservation Priority Index)', color='white', fontsize=10)
ax.set_title('CPI by Literature Source\n(error bars = min–max range)',
             color='white', fontweight='bold')
ax.tick_params(colors='white'); ax.spines[:].set_color('#444')
ax.grid(True, alpha=0.12, color='white', axis='x')
for i, row in enumerate(src_cpi.itertuples()):
    ax.text(row.mean+0.05, i, f"{row.mean:.2f} (n={int(row.count)})",
            va='center', color='white', fontsize=8.5)

fig.suptitle('Multi-Taxon Biodiversity Analysis — Alluri Sitaramaraju District\n'
             'Integrated from 5 Literature Sources · GEE Landsat 8/9',
             color='white', fontsize=14, fontweight='bold')
plt.savefig('ASR_GEE_Outputs/GEE_F5_Biodiversity_Dashboard.png',
            dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("  Saved: GEE_F5_Biodiversity_Dashboard.png")


# ══════════════════════════════════════════════════════════════════════════════

🎨 Generating Figure 5 — Taxon biodiversity dashboard...


NameError: name 'plt' is not defined

## 🌐 Cell 16 — Interactive geemap Viewer
Displays NDVI layers (2018/2020/2024), dNBR, district boundary, and all species points by IUCN status. Renders inline + saves as HTML.

In [ ]:
# SECTION F — GEEMAP INTERACTIVE MAP
# ══════════════════════════════════════════════════════════════════════════════
print("\n🗺️  Building interactive geemap...")

try:
    Map = geemap.Map(center=[cy, (bounds['xmin']+bounds['xmax'])/2], zoom=9)

    # Add NDVI layers
    for yr in [2018, 2020, 2024]:
        img   = compute_indices(get_landsat_collection(yr))
        ndvi  = img.select('NDVI')
        viz   = {'min': 0.2, 'max': 0.85, 'palette': ['red','yellow','green']}
        Map.addLayer(ndvi, viz, f'NDVI {yr}', shown=(yr==2024))

    # Add dNBR disturbance
    try:
        dnbr_viz = {'min': -0.1, 'max': 0.5,
                    'palette': ['00FF00','FFFF00','FF8C00','FF0000','8B0000']}
        Map.addLayer(dNBR, dnbr_viz, 'dNBR Disturbance 2020', shown=False)
    except Exception:
        pass

    # Add district boundary
    Map.addLayer(ASR_FC.style(color='FFD700', fillColor='00000000', width=2),
                 {}, 'ASR District Boundary')

    # Add species points
    for st in ORDERS:
        sub = threatened[threatened['status'] == st]
        col_hex = STATUS_COLORS[st].replace('#','')
        pts = ee.FeatureCollection([
            ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]),
                       {'species': row['common'], 'status': row['status'],
                        'group': row['group'], 'CPI': row['cpi']})
            for _, row in sub.iterrows()
        ])
        Map.addLayer(
            pts.style(color=col_hex, fillColor=col_hex+'aa',
                      pointSize=6, pointShape='circle'),
            {}, f'{STATUS_FULL[st]} ({len(sub)} spp)')

    Map.add_colorbar(
        {'min':0.2,'max':0.85,'palette':['red','yellow','green']},
        label='NDVI',
        layer_name='NDVI 2024'
    )

    Map.save('ASR_GEE_Outputs/ASR_Species_Interactive_Map.html')
    print("  Saved: ASR_Species_Interactive_Map.html")

except Exception as e:
    print(f"  geemap error: {e}")


# ══════════════════════════════════════════════════════════════════════════════


🗺️  Building interactive geemap...
  Saved: ASR_Species_Interactive_Map.html


## 📤 Cell 17 — Export All Data to CSV

In [ ]:
# SECTION G — EXPORT ALL DATA TO CSV
# ══════════════════════════════════════════════════════════════════════════════
print("\n📤 Exporting data files...")

# Species master table
export_cols = ['id','name','common','group','source','status','sensitivity',
               'habitat','lat','lon','forest_dep','dist_loss','recovery',
               'encounter','cites','endemic','trophic','cpi','iucn_num']
if 'gee_dist_loss' in df_species.columns:
    export_cols += ['gee_dist_loss','gee_recovery','gee_dnbr']

df_species[export_cols].sort_values('cpi', ascending=False) \
    .to_csv('ASR_GEE_Outputs/ASR_species_master.csv', index=False)

# NDVI statistics
df_ndvi_stats.to_csv('ASR_GEE_Outputs/ASR_district_ndvi_stats.csv', index=False)

# Recovery regression
if not df_recovery.empty:
    df_recovery.to_csv('ASR_GEE_Outputs/ASR_species_recovery_regression.csv', index=False)

# Species time series
if not df_sp_ts.empty:
    df_sp_ts.to_csv('ASR_GEE_Outputs/ASR_species_ndvi_timeseries.csv', index=False)

print("  Exported:")
for f in sorted(os.listdir('ASR_GEE_Outputs')):
    size = os.path.getsize(f'ASR_GEE_Outputs/{f}')
    print(f"    📄 {f}  ({size//1024} KB)")


# ══════════════════════════════════════════════════════════════════════════════


📤 Exporting data files...
  Exported:
    📄 ASR_Species_Interactive_Map.html  (106 KB)
    📄 ASR_district_ndvi_stats.csv  (0 KB)
    📄 ASR_species_master.csv  (4 KB)
    📄 ASR_species_ndvi_timeseries.csv  (5 KB)
    📄 ASR_species_recovery_regression.csv  (0 KB)
    📄 GEE_F1_NDVI_TimeSeries.png  (227 KB)
    📄 GEE_F2_Species_Distribution_Map.png  (384 KB)
    📄 GEE_F3_Disturbance_Analysis.png  (209 KB)
    📄 GEE_F4_Recovery_Priority.png  (260 KB)
    📄 GEE_F5_Biodiversity_Dashboard.png  (305 KB)


## ✅ Cell 18 — Analysis Summary

In [ ]:
cr_list = df_species[df_species['status']=='CR']['common'].tolist()
en_list = df_species[df_species['status']=='EN']['common'].tolist()

print("="*70)
print(" ANALYSIS COMPLETE")
print("="*70)
summary = [
    f"  District   : Alluri Sitaramaraju, Eastern Ghats, AP",
    f"  Boundary   : ASR_boundary.shp  ({TOTAL_AREA_KM2:.0f} km2)",
    f"  GEE Project: seismic-envoy-485304-p9",
    f"  Satellite  : Landsat 8/9 OLI - 30m - Nov-Feb dry-season composites",
    "",
    f"  SPECIES (5 sources)",
    "  " + "-"*58,
    f"  Total: {len(df_species)}  |  Threatened: {len(threatened)}",
    f"  CR ({len(cr_list)}): {', '.join(cr_list)}",
    f"  EN ({len(en_list)}): {', '.join(en_list)}",
    "",
    "  STATISTICS",
    "  " + "-"*58,
    f"  Mann-Kendall  : z={mk['z']:.4f}, p={mk['p']:.4f}  trend={mk['trend']}",
    f"  Spearman rho  : {rho:.4f},  p={pval:.4f}",
    f"  Kruskal-Wallis: H={H:.4f},  p={p_kw:.4f}",
    "",
    "  OUTPUTS -> ASR_GEE_Outputs/",
    "  " + "-"*58,
    "  GEE_F1_NDVI_TimeSeries.png",
    "  GEE_F2_Species_Distribution_Map.png",
    "  GEE_F3_Disturbance_Analysis.png",
    "  GEE_F4_Recovery_Priority.png",
    "  GEE_F5_Biodiversity_Dashboard.png",
    "  ASR_Species_Interactive_Map.html",
    "  ASR_species_master.csv",
    "  ASR_district_ndvi_stats.csv",
    "  ASR_species_recovery_regression.csv",
    "  ASR_species_ndvi_timeseries.csv",
]
for line in summary:
    print(line)
print("="*70)

 ANALYSIS COMPLETE
  District   : Alluri Sitaramaraju, Eastern Ghats, AP
  Boundary   : ASR_boundary.shp  (12629 km2)
  GEE Project: seismic-envoy-485304-p9
  Satellite  : Landsat 8/9 OLI - 30m - Nov-Feb dry-season composites

  SPECIES (5 sources)
  ----------------------------------------------------------
  Total: 31  |  Threatened: 31
  CR (4): Beddome's Cycad, Lantern Flower, Gharial, Red-crowned Roofed Turtle
  EN (8): Bengal Tiger, Asian Elephant, Dhole, Indian Pangolin, Sarai (E.Ghats Endemic), Wild Jamun (Endemic), Purple Frog, Mahseer

  STATISTICS
  ----------------------------------------------------------
  Mann-Kendall  : z=0.3004, p=0.7639  trend=increasing
  Spearman rho  : -0.3521,  p=0.3184
  Kruskal-Wallis: H=3.2667,  p=0.0707

  OUTPUTS -> ASR_GEE_Outputs/
  ----------------------------------------------------------
  GEE_F1_NDVI_TimeSeries.png
  GEE_F2_Species_Distribution_Map.png
  GEE_F3_Disturbance_Analysis.png
  GEE_F4_Recovery_Priority.png
  GEE_F5_Biodiversit